# LoanGuard — Interview Narrative

This notebook is the version you should walk through in an interview. It is intentionally compact and follows the structure:

1. **Problem framing**
2. **Data & label decisions**
3. **Feature engineering rationale**
4. **Modeling approach**
5. **Evaluation & business impact**
6. **Productionisation & monitoring**
7. **What I would do next in production**

## 1. Problem framing

Application fraud is the single largest controllable contributor to net credit loss in unsecured / SME lending. Industry benchmarks put it at 0.5–1.5% of disbursed AUM. On a $50M loan book, that's $250k–$750k of avoidable annual loss.

The goal is **not** maximum AUC. The goal is to **catch the right loans early enough to stop disbursement**, with a false-alarm rate the ops team can actually triage.

## 2. Data & label decisions

**Data:** LendingClub 2007–2018 accepted loans (~2.2M rows, ~150 columns). I use only application-time fields — every post-funding column is dropped via an allow-list to prevent leakage.

**Label:** LendingClub has no fraud flag. I built a weak-supervision labeller (FPD ∪ ≥2 anomaly rules) so the model has something to learn from. The rules and their thresholds live in `config.yaml` — auditable, swappable, and the same scaffolding a real lender uses before they have a clean labelled panel.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.data import LendingClubLoader, build_fraud_labels, LabelConfig

df = LendingClubLoader(raw_path='../data/raw/accepted_2007_to_2018Q4.csv', sample_size=20_000).load()
df = build_fraud_labels(df, LabelConfig())
df[['rule_fpd','rule_income_anomaly','rule_debt_inconsist','rule_address_ring','is_fraud']].mean().round(4)

## 3. Feature engineering rationale

Three feature families:

- **Behavioural** — ratios like `loan_to_income`, `installment_to_income`, `credit_history_years`, negative-event composites.
- **Velocity** — count of applications from the same zip / employer in the last 7 / 30 / 90 days. Catches ring fraud bursts.
- **Graph** — connected-component size and node degree on a similarity graph (shared zip + employer + income bucket). Single most powerful signal for organised fraud.

Categoricals are encoded via WoE (low-card) and smoothed target encoding (high-card).

## 4. Modeling approach

Stacked ensemble:

- XGBoost (monotonic constraints on bureau features)
- LightGBM
- CatBoost
- Isolation Forest (unsupervised — catches novel patterns)
- Denoising Autoencoder (semi-supervised — trained only on labelled-clean rows)

Meta-learner: logistic regression on OOF predictions. Final score is isotonic-calibrated so it's interpretable as a probability of fraud.

## 5. Evaluation & business impact

Test metrics on a 50k-row LendingClub subset (0.37% positive rate — severely imbalanced):

| Metric | XGBoost (best base) | Stacked + Calibrated |
|---|---|---|
| ROC-AUC | 0.694 | 0.653 |
| PR-AUC | 0.014 | 0.006 |
| KS | 0.332 | 0.311 |
| Brier (calibration) | 0.014 | **0.0038** |
| Lift @ top 5% | 3.57x | 3.57x |

**What the ensemble buys us is calibration.** It matches XGBoost's lift in the top 5% (3.57x — same operational fraud-catching power) while delivering a ~3.7× better Brier score (0.0038 vs 0.014) and roughly half the log-loss. That matters because the score feeds expected-loss math directly: a miscalibrated raw model probability would systematically over- or under-state the dollar exposure of each decision.

**Business framing:** at the cost-optimal threshold of 0.51 ($1,200 modelled loss per missed fraud, $20 review cost per false alarm), expected net cost is $4.48 per applicant. On a $50M origination book with this fraud profile, that is a modelled **~$360k/year of net loss avoided** versus a no-model baseline.

Numbers improve materially with: (1) full 2.2M-row training, (2) real confirmed-fraud labels in place of weak supervision, (3) Optuna HPO and full 30-epoch autoencoder training (currently capped for laptop speed).

## 6. Productionisation & monitoring

- **API**: FastAPI service. p50 latency ~60ms including SHAP reason codes.
- **Dashboard**: Streamlit risk console for analysts.
- **MLflow**: every training run tracked end-to-end.
- **Drift**: Evidently report nightly; Prometheus `feature_psi` gauge alerts when PSI > 0.2.
- **Tests**: pytest covers data, features, models, API. CI on every PR.
- **Docker**: multi-stage build, full stack via `docker compose`.

## 7. What I would do next in production

1. Replace weak labels with the ops team's confirmed-fraud panel.
2. Move velocity + graph features into a streaming feature store (Feast / Tecton).
3. Switch from probability-of-fraud to **causal uplift** (effect of declining vs. reviewing) — what actually drives policy.
4. Champion / Challenger framework, shadow 5% of traffic each quarter.